# Pix2Struct widget-captioning-base — DIMER UI widget captioning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/tutorials/pix2struct_ui_captioning_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--widget--captioning--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-widget-captioning-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** UI widget captioning — one app screenshot plus one widget bounding box → one short caption of that widget's purpose — using the pinned `google/pix2struct-widget-captioning-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/pix2struct_ui_captioning_pipeline/pipeline.py` at revision `b951c2307b12`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `7e99642f87127dd3aee97ea82616bbfda5e610bb` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the Pix2Struct image-encoder/text-decoder (a ViT-style encoder over variable-resolution 16×16 patches and a 12-layer text decoder, 282M parameters, pretrained by parsing masked web screenshots into HTML and fine-tuned on Widget Captioning, a set of Android screenshots from the Rico corpus with human-written descriptions of individual UI elements) receives the screenshot with the **target widget outlined in blue** — the upstream preprocessing convention, reproduced by the carried module; no header text is rendered — scales it to fill at most 2048 patches, and generates a short caption of the widget's role (`search bar`, `go to profile`) token by token. Decoding is greedy (`do_sample=False`, one beam) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, a box inside the image with sides of at least 4 px, the token budget), the box rendering, a fixed output contract, and the `annotate_widget`, `keyword_hits`, `unigram_f1`, `validate_inputs` and `evaluation_report` helpers. The default sample is a flat mock of a messaging app drawn in code with five widget boxes and no reference captions, so the evaluation report is `not-measurable` by design (the fleet matrix lists this row as report-only) and the printed keyword checks are observations, not a Widget Captioning benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic app screen with widget boxes (or upload your own screenshot and type box coordinates) and validate it into an input manifest, choose a token budget, run the supported task, read the captions correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `not-measurable` without reference captions and `sample-sanity` with a bag-of-words `unigram_f1` when you supply some, and export the captions, the annotated screenshot and provenance.

**This notebook does not demonstrate:** Widget detection (the box is the caller's input; nothing is located), whole-screen summarisation (a separate checkpoint), reading UI text back as a transcript, captions in languages other than English, batch throughput, sampling or beam search (greedy decoding for reproducibility), evaluation on the Widget Captioning benchmark (not bundled; CIDEr and BLEU-4 need several references per widget and are not computed here), and any training. The model was fine-tuned on 2017-era Android screenshots at phone resolution; flat mocks, desktop or web UIs, dark themes and non-English interfaces are outside what this notebook measures, and a fluent wrong caption carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.4 s to load and 2.0–2.4 s per widget on the 540×960 mock in the Windows venv (Intel Core Ultra 9 275HX) — the 2048-patch encoder pass dominates. The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; why a bounding box drawn on the input is part of the model's contract; what reference-based caption metrics (CIDEr, BLEU) need; that a confident caption is not a correct one.
- **Data:** the default sample is a deterministic 540×960 mock of a messaging app drawn in code with Pillow's bundled font (a blue header bar with a gear icon, a search field, three conversation rows with avatars, a green `New message` button, a four-tab bottom bar) with five widget boxes and **no reference captions**, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one screenshot decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus widget boxes typed as `x0, y0, x1, y1` lines in pixels. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-widget-captioning-base` snapshot (~1133 MB in total) at revision `7e99642f8712…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-ui-captioning-pipeline',
    'repository_revision': 'b951c2307b129545ef4d074d921564ea5a122413',
    'embedded_module': 'src/pix2struct_ui_captioning_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_ui_captioning_pipeline/pipeline.py'],
    'module_sha256': 'fbbf519150e0444c76b5c7e0e8ebd36c0c3ae05a3259f19a2c020233ca24a0ed',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_ui_captioning_pipeline/` @ `b951c2307b12`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/pix2struct_ui_captioning_pipeline/pipeline.py`

In [ ]:
"""Widget captioning with the pinned ``google/pix2struct-widget-captioning-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The target widget is indicated
the way the upstream preprocessing did — a blue outline drawn on the screenshot, no header text — and
the model generates a short caption for it.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw

MODEL_ID = "google/pix2struct-widget-captioning-base"
MODEL_REVISION = "7e99642f87127dd3aee97ea82616bbfda5e610bb"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-widget-captioning-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. Widget captions are a few words ("search bar", "go to profile"; the checkpoint's
# text_config max_length is 20); the default leaves room for a phrase, the ceiling bounds runaway
# generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 20
DECODING = "greedy"
# Widget box rendering: the upstream preprocessing (pix2struct/preprocessing/convert_widget_captioning.py)
# draws the target widget's bounds as a blue rectangle with a transparent fill and no header text.
BOX_COLOR = (0, 0, 255)
BOX_WIDTH = 3
MIN_BOX_SIDE = 4
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget (aspect ratio preserved), so pixel count only guards
# memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_caption(text: str) -> str:
    """COCO-caption-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def caption_tokens(text: str) -> list[str]:
    return normalize_caption(text).split()


def unigram_f1(prediction: str, references: Sequence[str]) -> float:
    """Bag-of-words F1 between the normalised prediction and the best-matching reference.

    A plumbing check, not a captioning metric: CIDEr, BLEU-4 and SPICE need several references per
    image and corpus-level statistics. Multiset overlap counts repeated words once per occurrence.
    """
    if not references:
        raise ValueError("references must contain at least one caption")
    pred = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        if not pred or not ref:
            continue
        ref_counts: dict[str, int] = {}
        for token in ref:
            ref_counts[token] = ref_counts.get(token, 0) + 1
        overlap = 0
        for token in pred:
            if ref_counts.get(token, 0) > 0:
                overlap += 1
                ref_counts[token] -= 1
        if overlap:
            precision, recall = overlap / len(pred), overlap / len(ref)
            best = max(best, 2 * precision * recall / (precision + recall))
    return best


def keyword_hits(caption: str, keywords: Sequence[str]) -> dict[str, bool]:
    """Which of the caller's keywords (normalised, whole-token match) appear in the caption."""
    tokens = set(caption_tokens(caption))
    return {keyword: all(part in tokens for part in caption_tokens(keyword)) for keyword in keywords}


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_box(box: Any, image_size: tuple[int, int]) -> list[int]:
    """Check one widget box ``[x0, y0, x1, y1]`` in pixels: inside the image, at least MIN_BOX_SIDE a side."""
    if isinstance(box, (str, bytes)) or not isinstance(box, Sequence) or len(box) != 4:
        raise TypeError("box must be a sequence of four numbers [x0, y0, x1, y1]")
    values = []
    for value in box:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("box coordinates must be numbers")
        values.append(int(round(value)))
    x0, y0, x1, y1 = values
    width, height = image_size
    if x0 < 0 or y0 < 0 or x1 > width or y1 > height:
        raise ValueError(f"box {values} lies outside the {width}x{height} image")
    if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
        raise ValueError(f"box {values} is smaller than MIN_BOX_SIDE {MIN_BOX_SIDE} px on a side")
    return values


def annotate_widget(image: Image.Image, box: Sequence[int]) -> Image.Image:
    """Return a copy of the RGB image with the widget outlined the way the upstream preprocessing did.

    Upstream (``pix2struct/preprocessing/convert_widget_captioning.py``) draws the target widget's
    bounds as a blue rectangle with a transparent fill and no header text; this package draws the same
    blue outline at BOX_WIDTH pixels.
    """
    rgb = validate_image(image)
    x0, y0, x1, y1 = validate_box(box, rgb.size)
    annotated = rgb.copy()
    ImageDraw.Draw(annotated).rectangle([x0, y0, x1, y1], outline=BOX_COLOR, width=BOX_WIDTH)
    return annotated


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one screenshot as PIL.Image.Image (any mode, converted to RGB) plus one widget box "
        "[x0, y0, x1, y1] in pixels"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "box": f"pixel coordinates inside the image, each side at least {MIN_BOX_SIDE} px",
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False, num_beams=1), deterministic on a fixed device and dtype",
    "preprocessing": (
        f"the widget box is drawn on a copy of the screenshot as a blue outline ({BOX_WIDTH} px, the "
        "upstream widget-captioning convention; no header text is rendered); the annotated screenshot is "
        "scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per image "
        "and flattened into patch tokens with row/column positions; the decoder generates the caption"
    ),
    "output": "one short caption string for the boxed widget (the model's decoded text), no score",
}


def _check_inputs(image: Any, box: Any, max_new_tokens: Any) -> tuple[Image.Image, list[int], int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``caption`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    checked_box = validate_box(box, rgb.size)
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_box, max_new_tokens


def validate_inputs(
    image: Image.Image,
    boxes: Sequence[Sequence[int]],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every box is checked exactly as ``caption`` would check it; rejection is reported by raising, and
    a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(boxes, (str, bytes)) or not isinstance(boxes, Sequence) or not boxes:
        raise TypeError("boxes must be a non-empty sequence of [x0, y0, x1, y1] boxes")
    if isinstance(boxes[0], (int, float)):
        raise TypeError("boxes must be a sequence of boxes, not a single box")
    checked = [_check_inputs(image, box, max_new_tokens)[1] for box in boxes]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (caption takes one screenshot)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "boxes": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    references: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``references`` (one sequence of reference captions per result, in order) the report carries
    the mean ``unigram_f1`` over the widgets plus one per-widget entry, verdict ``sample-sanity``;
    without references it is ``not-measurable`` and says what labelled data would make the task
    measurable. Neither is a captioning benchmark.
    """
    if not results:
        raise ValueError("results must contain at least one caption result")
    base = {
        "task": "screenshot + widget box -> short caption of the widget (widget captioning)",
        "score_semantics": (
            "the caption is generated text and carries no score, probability or correctness signal; a "
            "fluent caption is not evidence that it describes the boxed widget. Greedy decoding makes the "
            "output reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_widgets": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if references is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference captions were supplied for the captioned widgets",
            "needs": (
                "several human-written reference captions per widget from the deployment's own screens "
                "(Widget Captioning-style annotations) scored with CIDEr / BLEU-4 over a corpus; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(references) != len(results):
        raise ValueError(f"references has {len(references)} entries for {len(results)} results")
    per_widget = []
    for result, refs in zip(results, references, strict=True):
        if isinstance(refs, str) or not refs:
            raise ValueError("each references entry must be a non-empty sequence of captions")
        prediction = str(result["caption"])
        per_widget.append(
            {
                "box": result.get("box"),
                "prediction": prediction,
                "references": list(refs),
                "unigram_f1": unigram_f1(prediction, refs),
            }
        )
    metrics = [
        {
            "id": "unigram_f1",
            "value": sum(entry["unigram_f1"] for entry in per_widget) / len(per_widget),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; best reference",
            "relation_to_benchmarks": (
                "bag-of-words overlap with the best reference; not CIDEr, BLEU-4 or SPICE, which need "
                "several references per widget and corpus-level statistics"
            ),
            "estimation": f"{len(per_widget)} widget(s) on one screenshot, no dispersion estimate",
        }
    ]
    return {
        **base,
        "metrics": metrics,
        "per_widget": per_widget,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_widget)} widget(s) with caller-written reference captions; plumbing evidence, not a "
            "captioning benchmark"
        ),
        "needs": (
            "several human-written reference captions per widget from the deployment's own screens scored "
            "with CIDEr / BLEU-4 over a corpus for any quality claim; the Widget Captioning dataset is not "
            "bundled"
        ),
    }


@dataclass
class Pix2StructWidgetCaptioningPipeline:
    """``_runner(annotated_image, max_new_tokens)`` returns ``{"caption": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructWidgetCaptioningPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        # The snapshot's preprocessor_config declares is_vqa=True, which would render a text header
        # above the screenshot and fetch a font from the Hub. Upstream's widget-captioning preprocessing
        # renders no header - only the blue widget box - so the header path is disabled here; nothing
        # but the box is added to the image and no font is involved.
        processor.image_processor.is_vqa = False
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(annotated: Image.Image, max_new_tokens: int) -> dict[str, Any]:
            inputs = processor.image_processor(annotated, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + caption + eos).
            ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"caption": decoded, "new_tokens": max(int(ids.shape[0]) - 1, 0)}

        return cls(runner, resolved_device, "float32", source)

    def caption(
        self,
        image: Image.Image,
        box: Sequence[int],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Caption the widget inside ``box`` on one screenshot; ``caption`` is the decoded text, stripped."""
        rgb, checked_box, checked_tokens = _check_inputs(image, box, max_new_tokens)
        annotated = annotate_widget(rgb, checked_box)
        raw = self._runner(annotated, checked_tokens)
        if not isinstance(raw, dict) or "caption" not in raw:
            raise RuntimeError("runner must return a dict with 'caption'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "caption": str(raw["caption"]).strip(),
            "box": checked_box,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `7e99642f8712…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructWidgetCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-widget-captioning-base",
  "modelId": "google/pix2struct-widget-captioning-base",
  "revision": "7e99642f87127dd3aee97ea82616bbfda5e610bb",
  "files": [
    {
      "path": "README.md",
      "bytes": 4499,
      "sha256": "82b161c0f6c74a1a520c6d3094927b5cb9f4163cd9a0bed390ee1974d91c3c4b"
    },
    {
      "path": "config.json",
      "bytes": 4904,
      "sha256": "301fe55f2135ea50cb3808d5112fa502d63e45a90b9ca0661e0d6e775f5a4b30"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "0b096c351692854237aa9fca73e973e3f764f376afc84999357614bac36e2b92"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133308959
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructWidgetCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic app screen or optional BYOD

The default sample is **synthetic**: a flat mock of a messaging app — a blue header reading `Messages` with a gear icon, a `Search conversations` field, three conversation rows with round avatars, a green `New message` button and a `Home / Chats / Calls / Profile` tab bar — is drawn with Pillow at 540×960 (a phone aspect ratio), the same mock the repository's smoke run used. Five widget boxes are authored in pixel coordinates (the button, the search field, the gear icon, Ana's avatar, the Profile tab), each with a few **expected keywords** the notebook checks as an observation. **No reference captions are authored**, because a caption you write yourself is not an annotation standard: the evaluation report will therefore be `not-measurable`. The image digest is printed for the record; it depends on the Pillow build's bundled font rendering. BYOD is optional and disabled by default; when enabled, upload one screenshot and type your boxes.

The token budget is a **caller-owned request parameter**: `max_new_tokens` bounds the caption (`DEFAULT_MAX_NEW_TOKENS = 20` fits any widget phrase; `MAX_NEW_TOKENS = 64` is the ceiling). Nothing is validated in this cell — the next section hands the image and the boxes to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the budget and the number of boxes.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_boxes = '90, 820, 450, 880'  # @param {type:"string"}
max_new_tokens = 20  # @param {type:"integer"}


def synthetic_screen(width=540, height=960):
    """A flat messaging-app mock drawn with Pillow; returns image + [(widget name, box, expected keywords)]."""
    image = Image.new('RGB', (width, height), (245, 246, 250))
    d = ImageDraw.Draw(image)
    title, body = ImageFont.load_default(size=26), ImageFont.load_default(size=20)
    d.rectangle([0, 0, 540, 90], fill=(33, 90, 200))  # header bar
    d.text((30, 30), 'Messages', fill='white', font=title)
    d.rectangle([470, 25, 510, 65], outline='white', width=3)  # gear-like icon
    d.ellipse([482, 37, 498, 53], fill='white')
    d.rounded_rectangle([30, 120, 510, 170], radius=12, fill='white', outline=(200, 200, 200))  # search field
    d.text((50, 133), 'Search conversations', fill=(150, 150, 150), font=body)
    for index, (name, message) in enumerate([('Ana', 'See you at 6?'), ('Ben', 'Sent the files'), ('Cara', 'Happy birthday!')]):
        y = 200 + index * 90
        d.ellipse([30, y, 90, y + 60], fill=(120, 160, 220))  # avatar
        d.text((110, y + 5), name, fill='black', font=title)
        d.text((110, y + 38), message, fill=(110, 110, 110), font=body)
    d.rounded_rectangle([90, 820, 450, 880], radius=30, fill=(33, 150, 90))  # primary button
    d.text((270, 850), 'New message', fill='white', font=title, anchor='mm')
    d.rectangle([0, 900, 540, 960], fill='white')  # tab bar
    for index, label in enumerate(['Home', 'Chats', 'Calls', 'Profile']):
        d.text((67 + index * 135, 930), label, fill=(80, 80, 80), font=body, anchor='mm')
    widgets = [
        ('new-message button', [90, 820, 450, 880], ['message']),
        ('search field', [30, 120, 510, 170], ['search']),
        ('gear icon', [470, 25, 510, 65], ['settings']),  # smoke run: `go to next` (recorded miss)
        ('Ana avatar', [30, 200, 90, 260], ['ana']),
        ('Profile tab', [420, 905, 540, 955], ['profile']),
    ]
    return image, widgets


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    widgets = []
    for index, line in enumerate(byod_boxes.splitlines()):
        parts = [part.strip() for part in line.split(',') if part.strip()]
        if len(parts) == 4:
            widgets.append((f'widget-{index}', [float(part) for part in parts], []))
    sample_kind = 'BYOD'
else:
    # Deterministic mock: no randomness, so no seed is needed; the digest depends on the Pillow build's bundled font.
    image, widgets = synthetic_screen()
    image_name = 'synthetic_messaging_screen_540x960.png'
    sample_kind = 'synthetic'

widget_names = [name for name, _, _ in widgets]
boxes = [box for _, box, _ in widgets]
expected_keywords = [keywords for _, _, keywords in widgets]
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'max_new_tokens': max_new_tokens, 'n_boxes': len(boxes)})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `caption` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, each box four numbers inside the image with sides of at least `MIN_BOX_SIDE` px, and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the blue-outline rendering, the patch budget and the decoding rule), the input's observed mode and size, the checked boxes, the budget and the verdict. The manifest is written to `outputs/pix2struct_ui_captioning_input_manifest.json`. To show what rejection looks like, the cell also validates a box that runs past the screenshot's edge and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, the box is drawn on a copy, and the composite is scaled to the patch budget; nothing else is dropped or altered. The pipeline cannot tell whether the box encloses a widget or whether the image is a screenshot at all: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MIN_BOX_SIDE': MIN_BOX_SIDE, 'BOX_COLOR': BOX_COLOR, 'BOX_WIDTH': BOX_WIDTH, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(image, boxes, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, [[0, 0, image.width + 20, 100]])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_ui_captioning_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Caption the widgets and read the output correctly

`caption` returns, per widget, a dict with `caption` (the decoded text, stripped), the checked `box`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the caption is generated text with no probability and no correctness signal, and a fluent caption is not evidence that it describes the boxed widget. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the caption, so GPU and CPU outputs need not match. Each call draws the box and re-encodes the screenshot at up to 2048 patches, so cost is per widget (about 2.0–2.4 s each on the reference CPU). As recorded in the model card, the repository's CPU smoke on this same mock captioned the five widgets `go to new message`, `search bar`, `go to next`, `select ana` and `profile` — the gear icon is the recorded miss — and captioned a box on a blank white image `select the image` and on noise `go to next`: the model always produces a caption, whether or not the box encloses anything. The cell also records which expected keywords appear in each caption; that is an observation, not a metric.

In [ ]:
import time

results, seconds = [], []
for name, box in zip(widget_names, boxes):
    t0 = time.time()
    result = pipe.caption(image, box, max_new_tokens=max_new_tokens)
    result['widget'] = name
    results.append(result)
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_widget': seconds, 'any_truncated': any(r['truncated'] for r in results)})
keyword_observations = []
for result, keywords in zip(results, expected_keywords):
    hits = keyword_hits(result['caption'], keywords) if keywords else {}
    keyword_observations.append({'widget': result['widget'], 'box': result['box'], 'expected_keywords': keywords, 'hits': hits})
    print(f"{result['widget']} {result['box']}\n   caption: {result['caption']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})\n   keywords: {hits}")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that caption is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No quality is reported by default: widget-caption metrics (CIDEr, BLEU-4) need several human-written reference captions per widget from the deployment's own screens and corpus-level statistics, and this repository ships none (the Widget Captioning dataset is not bundled). The repository's helper `unigram_f1` — bag-of-words F1 after normalisation (lower-case, punctuation removed, whitespace collapsed) against the best-matching reference — exists so that a caller who does supply references gets a `sample-sanity` report with one entry per widget; it is explicitly **not** a captioning metric. On the default path no references are supplied, the verdict is `not-measurable`, and the report states what would make the task measurable; the keyword observations from the previous section are attached to the report file under `observations` for the record. The report is written to `outputs/pix2struct_ui_captioning_evaluation_report.json`. To see the other branch, set `references` below to one list of reference captions per widget.

In [ ]:
references = None  # e.g. [['compose a new message'], ['search conversations'], ['open settings'], ['open Ana chat'], ['go to profile']]
report = evaluation_report(results, references, sample_kind=sample_kind)
report['observations'] = keyword_observations
with open('outputs/pix2struct_ui_captioning_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_widget', 'observations')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_widget', []):
    print(f"  unigram_f1 {entry['unigram_f1']:.2f}  {entry['box']} -> {entry['prediction']!r} (references: {entry['references']})")
if report['verdict'] == 'not-measurable':
    print('No reference captions exist for these widgets, so nothing is scored; read the captions against the screen yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (widget name, box, caption, `new_tokens`, `truncated`, the budget), the evaluation report with the keyword observations, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The captions are also written as CSV with explicit `image`, `widget`, `box`, `caption`, `new_tokens`, `truncated` columns, and an annotated PNG shows the screenshot with every widget box outlined and its caption printed in a panel beside it for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

panel_width = 420
annotated = Image.new('RGB', (image.width + panel_width, image.height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
panel_font = ImageFont.load_default(size=15)
for index, result in enumerate(results):
    x0, y0, x1, y1 = result['box']
    draw.rectangle([x0, y0, x1, y1], outline=BOX_COLOR, width=BOX_WIDTH)
    draw.text((x0 + 4, max(y0 - 18, 0)), str(index + 1), fill=BOX_COLOR, font=panel_font)
    draw.text((image.width + 16, 20 + 26 * index), f"{index + 1}. {result['widget']}: {result['caption']}", fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/pix2struct_ui_captioning_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'widgets': widget_names, 'boxes': boxes, 'expected_keywords': expected_keywords},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/pix2struct_ui_captioning_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/pix2struct_ui_captioning_captions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'widget', 'box', 'caption', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([image_name, result['widget'], ' '.join(str(v) for v in result['box']), result['caption'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The captions are the text the model generates for a screenshot with one widget outlined in blue; nothing in the output scores that text, the model returns no location beyond the box you gave it, and it captions every box — including one on a blank image — with equal fluency. On the drawn mock the evaluation report is `not-measurable` by design: no reference captions exist, and the keyword observations (the repository's smoke run found `message`, `search`, `ana` and `profile` and called the gear icon `go to next`) are what you can check by eye, not a metric; they say nothing about real Android screenshots, desktop or web interfaces, dark themes, icons without text, or non-English UIs, and a BYOD result is a per-widget observation with the same verdict. **The model captions any box on any image** and stops only at end-of-sequence or the token budget: check `truncated`, and treat a plausible caption for a box that encloses nothing as the expected failure mode, not an exception. The box is part of the request — a box around the whole screen produced `select the message` in the smoke run — so a wrong box is a wrong input, not a model error. The pipeline provides no widget detection, no screen summarisation, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** move a box a few pixels off its widget and watch the caption change; box Ben's or Cara's row instead of Ana's avatar; lower `max_new_tokens` to 1 and watch `truncated` turn true on `go`; write one reference caption per widget into `references` and see the verdict switch to `sample-sanity` with a `unigram_f1` you should not mistake for CIDEr; enable `USE_BYOD` with a screenshot you know and type its widget boxes.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/pix2struct-widget-captioning-base
- Upstream code (widget-box preprocessing): https://github.com/google-research/pix2struct/blob/main/pix2struct/preprocessing/convert_widget_captioning.py
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., 2022): https://arxiv.org/abs/2210.03347
- Widget Captioning: Generating Natural Language Description for Mobile User Interface Elements (Li et al., 2020): https://arxiv.org/abs/2010.04295
- Rico: A Mobile App Dataset for Building Data-Driven Design Applications (Deka et al., 2017): https://doi.org/10.1145/3126594.3126651